In [1]:
# Reload all modules to ensure latest versions
import importlib
import Manager_Agent
import Market_Expectation_Agent
import Revenue_Segmentation_Read_Agent
import Macro_Analyst_Agent

# Reload all modules
importlib.reload(Manager_Agent)
importlib.reload(Market_Expectation_Agent)
importlib.reload(Revenue_Segmentation_Read_Agent)
importlib.reload(Macro_Analyst_Agent)

# 2. Then import the functions
from Manager_Agent import quick_analysis, get_manager_result, get_manager_progress

# Test all three agents with your existing variables
import asyncio
import json
import asyncio
import yfinance as yf
from LLM_Call_Agent import LLMCallAgent
# pip install langchain-openai pydantic
from langchain_openai import ChatOpenAI
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import List

import News_Verification
importlib.reload(News_Verification)

/var/folders/f3/_yt94f597zx46t0j7p13pmfh0000gn/T/ipykernel_5707/2115152017.py:3: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  import Manager_Agent
/Users/xikinki/Desktop/Fintegrate_AI_File/Streamlit_APP/News_Verification.py:55: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`

<module 'News_Verification' from '/Users/xikinki/Desktop/Fintegrate_AI_File/Streamlit_APP/News_Verification.py'>

In [22]:
# Your query and ticker
user_question = "Federal Reserve Powell indicates conditions, 'may warrant' interest rate cuts as Fed proceeds 'carefully'. Provide comprehensive analysis of how this affects technology stocks."
ticker = "MSTR"
user_id = "123"

REDIS_CONFIG = {
    'host': 'redis-16376.crce197.us-east-2-1.ec2.redns.redis-cloud.com',
    'port': 16376,
    'password': 'rl8242B4UItBhFzgHW5APEqZnkYoaEZv'
}


## Step 1). News Verification

In [32]:


async def test_news_verification():
    # Test statement and user ID from your existing variables
    test_statement = user_question
    test_user_id = user_id
    
    print("🔍 Starting Enhanced News Verification Test...")
    print(f"📰 Statement: {test_statement}")
    print(f"👤 User ID: {test_user_id}")
    print("=" * 80)
    
    try:
        # Run the enhanced verification pipeline with user tracking
        result = await News_Verification.verify_statement_with_user(
            statement=test_statement,
            user_id=test_user_id,
            use_video=False
        )
        
        print("\n✅ Verification Complete!")
        print("=" * 80)
        print(f"📊 Final Decision: {result.final_decision}")
        print(f" Final Reasoning: {result.final_reasoning}")
        print(f" Reference Links: {len(result.reference_links) if result.reference_links else 0}")
        
        # Show filter results
        print("\n Filter Results:")
        for filter_result in result.filters:
            status_emoji = "✅" if filter_result.status.value == "passed" else "❌" if filter_result.status.value == "failed" else "⏭️"
            print(f"{status_emoji} {filter_result.name}: {filter_result.status.value}")
            if filter_result.details:
                print(f"   Details: {filter_result.details[:100]}...")
        
        # Test database retrieval
        print("\n💾 Testing Database Retrieval...")
        db = News_Verification.NewsVerificationDB(test_user_id)
        
        progress = db.get_progress()
        print(f"📈 Progress Data: {progress}")
        
        stored_result = db.get_result()
        print(f"📊 Stored Result Decision: {stored_result.get('final_decision', 'Not found')}")
        
        return result
        
    except Exception as e:
        print(f"❌ Error during verification: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# Run the test
print("🚀 Running Enhanced News Verification Test with Fixed Redis...")
verifcation_result = await test_news_verification()

🚀 Running Enhanced News Verification Test with Fixed Redis...
🔍 Starting Enhanced News Verification Test...
📰 Statement: Federal Reserve Powell indicates conditions, 'may warrant' interest rate cuts as Fed proceeds 'carefully'. Provide comprehensive analysis of how this affects technology stocks.
👤 User ID: 123
✅ Redis connected for user 123
📊 Progress Update: starting verification - started (0%)
2025-08-23 23:30:22,802 - INFO - Starting verification for user 123: Federal Reserve Powell indicates conditions, 'may ... (use_video=False)
📊 Progress Update: source verification - started (10%)
[DEBUG] Filter 1: Starting tavily search...
[DEBUG] tavily_search: Starting search with query: Federal Reserve Powell indicates conditions, 'may warrant' interest rate cuts as Fed proceeds 'carefully'. Provide comprehensive analysis of how this affects technology stocks.
[DEBUG] tavily_search: Created TavilySearch tool
[DEBUG] tavily_search: Raw result type: <class 'dict'>
[DEBUG] tavily_search: Raw r

## Step 2). Call Fitering New to Impaction if Not Noise to Investment

In [24]:


# Run the analysis with automatic multiprocessing
async def run_analysis():
    print("🚀 Starting Manager Agent Analysis...")
    
    # This ONE function does everything automatically:
    # 1. Analyzes your query
    # 2. Routes to appropriate agents
    # 3. Runs ALL agents in parallel with multiprocessing
    # 4. Returns complete results
    complete_results = await quick_analysis(
        user_query=user_question,
        ticker=ticker,
        user_id=user_id
    )
    
    if complete_results:
        print("\n✅ ANALYSIS COMPLETED SUCCESSFULLY!")
        print(f"📊 Total agents executed: {complete_results['execution_summary']['total_agents_executed']}")
        print(f"✅ Successful executions: {complete_results['execution_summary']['successful_executions']}")
        print(f"❌ Failed executions: {complete_results['execution_summary']['failed_executions']}")
        
        # Access individual agent results
        print("\n🔍 INDIVIDUAL AGENT RESULTS:")
        for agent_name, agent_result in complete_results['agent_results'].items():
            print(f"\n{agent_name}:")
            print(f"{'=' * 50}")
            if isinstance(agent_result, str) and agent_result.startswith("Error"):
                print(f"❌ {agent_result}")
            else:
                print(f"✅ {str(agent_result)[:300]}...")
        
        return complete_results
    else:
        print("\n❌ Analysis failed")
        return None



## Decision For Making

In [30]:
Call_Decision = verifcation_result.final_decision  

In [31]:
if Call_Decision == "Not Noise for Investment":
    impaction_results = await run_analysis()
    print(impaction_results)
else:
    print(f'Verification Decision: {Call_Decision}, No impaction on asset')

🚀 Starting Manager Agent Analysis...
✅ Manager Agent Frontend Redis connected: redis-16204.fcrce180.us-east-1-1.ec2.redns.redis-cloud.com:16204
2025-08-23 23:24:32,544 - INFO - 📥 Using OpenAI API key from Stock_Trend_Storage_Agent
2025-08-23 23:24:32,545 - INFO - 📥 Using DeepSeek API key from Stock_Trend_Storage_Agent
2025-08-23 23:24:32,566 - INFO - ✅ OpenAI client initialized
2025-08-23 23:24:32,583 - INFO - ✅ DeepSeek client initialized
2025-08-23 23:24:32,584 - INFO - 🤖 LLM Call Agent initialized
2025-08-23 23:24:32,584 - INFO -    - Default provider: deepseek
2025-08-23 23:24:32,584 - INFO -    - Default model: deepseek-chat
2025-08-23 23:24:32,584 - INFO -    - OpenAI: Enabled
2025-08-23 23:24:32,585 - INFO -    - DeepSeek: Enabled
🚀 MANAGER AGENT - AUTOMATIC MULTIPROCESSING ANALYSIS
📝 Query: Federal Reserve Powell indicates conditions, 'may warrant' interest rate cuts as Fed proceeds 'carefully'. Provide comprehensive analysis of how this affects technology stocks.
🎯 Ticker: MST

/Users/xikinki/anaconda3/envs/arviz_env/lib/python3.10/site-packages/langchain_openai/chat_models/base.py:1896: UserWarning: Received a Pydantic BaseModel V1 schema. This is not supported by method="json_schema". Please use method="function_calling" or specify schema via JSON Schema or Pydantic V2 BaseModel. Overriding to method="function_calling".
  warnings.warn(
/Users/xikinki/anaconda3/envs/arviz_env/lib/python3.10/site-packages/langchain_openai/chat_models/base.py:1896: UserWarning: Received a Pydantic BaseModel V1 schema. This is not supported by method="json_schema". Please use method="function_calling" or specify schema via JSON Schema or Pydantic V2 BaseModel. Overriding to method="function_calling".
  warnings.warn(


2025-08-23 23:24:33,136 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"

📊 ROUTING DECISIONS:
   Market Expectation: ✅ CALL
   Revenue Segmentation: ❌ SKIP
   Macro Analyst: ✅ CALL
📊 Frontend Progress Update: query_analysis - completed (20%) - Agent: Manager Agent

📋 STEP 2: Creating agent calling form...
📊 Frontend Progress Update: agent_planning - started (30%) - Agent: Manager Agent
✅ Manager Agent Frontend Redis connected: redis-16204.fcrce180.us-east-1-1.ec2.redns.redis-cloud.com:16204
2025-08-23 23:24:44,026 - INFO - 📥 Using OpenAI API key from Stock_Trend_Storage_Agent
2025-08-23 23:24:44,027 - INFO - 📥 Using DeepSeek API key from Stock_Trend_Storage_Agent
2025-08-23 23:24:44,046 - INFO - ✅ OpenAI client initialized
2025-08-23 23:24:44,065 - INFO - ✅ DeepSeek client initialized
2025-08-23 23:24:44,066 - INFO - 🤖 LLM Call Agent initialized
2025-08-23 23:24:44,067 - INFO -    - Default provider: deepseek
2025-08-23 23:24:44,067 - INFO -    -

/Users/xikinki/anaconda3/envs/arviz_env/lib/python3.10/site-packages/langchain_openai/chat_models/base.py:1896: UserWarning: Received a Pydantic BaseModel V1 schema. This is not supported by method="json_schema". Please use method="function_calling" or specify schema via JSON Schema or Pydantic V2 BaseModel. Overriding to method="function_calling".
  warnings.warn(


✅ Manager Agent Frontend Redis connected: redis-16204.fcrce180.us-east-1-1.ec2.redns.redis-cloud.com:16204
2025-08-23 23:24:44,282 - INFO - 📥 Using OpenAI API key from Stock_Trend_Storage_Agent
2025-08-23 23:24:44,282 - INFO - 📥 Using DeepSeek API key from Stock_Trend_Storage_Agent
2025-08-23 23:24:44,300 - INFO - ✅ OpenAI client initialized
2025-08-23 23:24:44,317 - INFO - ✅ DeepSeek client initialized
2025-08-23 23:24:44,318 - INFO - 🤖 LLM Call Agent initialized
2025-08-23 23:24:44,318 - INFO -    - Default provider: deepseek
2025-08-23 23:24:44,319 - INFO -    - Default model: deepseek-chat
2025-08-23 23:24:44,319 - INFO -    - OpenAI: Enabled
2025-08-23 23:24:44,320 - INFO -    - DeepSeek: Enabled
🚀 Starting TRUE PARALLEL Agent Execution...
📊 Frontend Progress Update: parallel_execution - started (65%) - Agent: Manager Agent
📊 Executing 2 agents in TRUE PARALLEL...
🚀 Launching ALL agents SIMULTANEOUSLY at 414477.89s...
🔄 Starting Market_Expectation_Agent at 414477.89s...
2025-08-23

/Users/xikinki/anaconda3/envs/arviz_env/lib/python3.10/site-packages/langchain_openai/chat_models/base.py:1896: UserWarning: Received a Pydantic BaseModel V1 schema. This is not supported by method="json_schema". Please use method="function_calling" or specify schema via JSON Schema or Pydantic V2 BaseModel. Overriding to method="function_calling".
  warnings.warn(


2025-08-23 23:24:44,645 - INFO - ✓ Ping successful - Redis server is reachable
2025-08-23 23:24:44,645 - INFO - ✓ Successfully connected to Redis
2025-08-23 23:24:44,646 - INFO - 📥 Using OpenAI API key from Stock_Trend_Storage_Agent
2025-08-23 23:24:44,647 - INFO - 📥 Using DeepSeek API key from Stock_Trend_Storage_Agent
2025-08-23 23:24:44,664 - INFO - ✅ OpenAI client initialized
2025-08-23 23:24:44,681 - INFO - ✅ DeepSeek client initialized
2025-08-23 23:24:44,682 - INFO - 🤖 LLM Call Agent initialized
2025-08-23 23:24:44,682 - INFO -    - Default provider: deepseek
2025-08-23 23:24:44,683 - INFO -    - Default model: deepseek-chat
2025-08-23 23:24:44,683 - INFO -    - OpenAI: Enabled
2025-08-23 23:24:44,683 - INFO -    - DeepSeek: Enabled
2025-08-23 23:24:44,684 - INFO - 🤖 Stock Trend Analyst Agent initialized
2025-08-23 23:24:44,684 - INFO -    - Redis: redis-16376.crce197.us-east-2-1.ec2.redns.redis-cloud.com:16376
2025-08-23 23:24:44,685 - INFO -    - Collection: Stock_Trend_INFOS
